# IOAI — 2025 Summer National Parametric Palette (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('petike.pth 는 노트북 첫 셀이 gdown 으로 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Parametric Palette (Paraméteres Paletta)

> **HAIO 2025 — Summer National Finals (CV / generative)**

Manipulate the latent space of the **frozen VAE `petike.pth`** so its decoder draws the targets (same *latent-optimisation* mechanism as Madarian Cow's "Magic"). Submit **15 latent vectors** (`submission.npy`, shape `(15, 48)`): rows 0–4 = **magenta circle**, 5–9 = **orange triangle**, 10–14 = **minimise blue**. Score = mean over the 3 subtasks of `shape_prob × colour_match` (threshold-free; colour measured on the decoded pixels).

⚠️ **The baseline below just *encodes* hand-drawn primary-colour shapes (no optimisation).** It gets the shapes right, but the VAE was only trained on red/blue/green — so it **cannot produce magenta** (magenta subtask ≈ 0). Use **latent optimisation (the "Magic")** to push the frozen decoder toward the out-of-distribution colours — see the model solution. (petike.pth auto-downloads via gdown.)

In [ ]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw
# petike.pth: 헤드리스에서 gdown 자동 다운로드(없으면)
if not os.path.exists('data/petike.pth') and not os.path.exists('petike.pth'):
    os.makedirs('data', exist_ok=True)
    try:
        import gdown
    except ImportError:
        import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','gdown'],check=True); import gdown
    gdown.download(id='1wuYIUV9rp-OeSWDSrTllmrTt50YzbxxP', output='data/petike.pth', quiet=True)
PETIKE='data/petike.pth' if os.path.exists('data/petike.pth') else 'petike.pth'
class Petike(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Conv2d(3,32,4,2,1),nn.BatchNorm2d(32),nn.ReLU(),nn.Conv2d(32,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.ReLU())
        self.flatten=nn.Flatten(); self.fc_mu=nn.Linear(128*4*4,48); self.fc_logvar=nn.Linear(128*4*4,48)
        self.decoder_input=nn.Linear(48,48)
        self.decoder=nn.Sequential(nn.Unflatten(1,(3,4,4)),nn.ConvTranspose2d(3,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.ConvTranspose2d(64,32,4,2,1),nn.BatchNorm2d(32),nn.ReLU(),nn.ConvTranspose2d(32,16,4,2,1),nn.BatchNorm2d(16),nn.ReLU(),nn.Conv2d(16,3,3,padding=1),nn.Sigmoid())
        self.register_buffer('shape_input',torch.zeros(48)); self.shape_input[0:24]=1.0
        self.register_buffer('color_input',torch.zeros(48)); self.color_input[24:48]=1.0
        for i in [0,12,24,36]: self.color_input[i]=0.0; self.shape_input[i]=0.0
        self.shape_classifier=nn.Sequential(nn.Linear(48,16),nn.ReLU(),nn.Linear(16,3))
        self.color_classifier=nn.Sequential(nn.Linear(48,16),nn.ReLU(),nn.Linear(16,3))
        self.size_regressor=nn.Sequential(nn.Linear(48,16),nn.ReLU(),nn.Linear(16,1))
        self.mask_head=nn.Sequential(nn.Linear(48,48),nn.ReLU(),nn.Linear(48,32*32),nn.Sigmoid())
    def encode(self,x): return self.fc_mu(self.flatten(self.encoder(x)))
    def decode(self,z): return self.decoder(self.decoder_input(z))
m=Petike(); m.load_state_dict(torch.load(PETIKE,map_location='cpu')); m.eval()
for p in m.parameters(): p.requires_grad_(False)
def draw(shape,color=(255,0,0)):
    img=Image.new('RGB',(32,32),(255,255,255)); d=ImageDraw.Draw(img)
    if shape=='circle': d.ellipse([8,8,24,24],fill=color)
    elif shape=='square': d.rectangle([9,9,23,23],fill=color)
    else: d.polygon([(16,7),(7,25),(25,25)],fill=color)
    return torch.tensor(np.array(img)/255.,dtype=torch.float).permute(2,0,1)[None]
sidx={sh:int(m.shape_classifier(m.encode(draw(sh))*m.shape_input).argmax()) for sh in ['triangle','square','circle']}
print('petike loaded · shape indices:', sidx)

In [ ]:
# NAIVE: encode plain shapes; decode reconstructs ~the trained colours (can't reach magenta)
specs = [('circle',(255,0,0))]*5 + [('triangle',(255,0,0))]*5 + [('square',(0,255,0))]*5
Z = []
torch.manual_seed(0)
for sh, col in specs:
    z = m.encode(draw(sh, col)).detach() + 0.01*torch.randn(1,48)   # tiny jitter for 5 distinct
    Z.append(z)
Z = torch.cat(Z, 0).numpy().astype('float32')
np.save('submission.npy', Z)
print('wrote submission.npy', Z.shape, '(naive — magenta will score ~0; optimise the latent to improve)')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)